# Numerical integrity preflight

**Purpose:** Check the prepared HDF5 structure, trial targets, retained cycles and participant-data join.

**Required inputs:** Prepared Baček HDF5; Bacek2023_subject_biological_data.csv.

**Outputs:** Source-integrity checks and trial ledger.

**Usage:** Run before source-dependent modelling. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 02 — Baček numerical-integrity preflight (no model training)
from pathlib import Path
import csv
import hashlib
import json
import platform
import zipfile

import h5py
import numpy as np

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
BIO = Path('Bacek2023_subject_biological_data.csv')
OUT = Path('02_Bacek_Numerical_Integrity_Preflight_results')
EXPECTED_SHA = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
OUTER_FOLDS = (
    ('S08', 'S10', 'S16', 'S21'),
    ('S11', 'S13', 'S17', 'S20'),
    ('S01', 'S02', 'S05', 'S18'),
    ('S03', 'S07', 'S12', 'S14', 'S19'),
    ('S04', 'S06', 'S09', 'S15'),
)
FOLD = {sid: i + 1 for i, group in enumerate(OUTER_FOLDS) for sid in group}


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def scalar_attr(trial, name):
    if name not in trial.attrs:
        return None
    values = np.asarray(trial.attrs[name]).reshape(-1)
    return values[0] if values.size == 1 else None


def real_numeric(array):
    return np.issubdtype(array.dtype, np.number) and np.isrealobj(array)


def save_csv(path, rows, columns):
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


if not H5.is_file() or not BIO.is_file():
    raise FileNotFoundError('Place this notebook alongside the corrected HDF5 and biological CSV.')

source_sha = sha256_file(H5)
if source_sha != EXPECTED_SHA:
    raise ValueError('HDF5 SHA-256 mismatch. Stop: do not inspect or substitute another dataset. '
                     f'Observed SHA-256: {source_sha}')

OUT.mkdir(exist_ok=True)
issues, ledger = [], []


def flag_problem(uid, check, explanation):
    issues.append({'trial_uid': uid, 'check': check, 'detail': str(explanation)})


# Inspect only T01–T15; read one angle matrix at a time, never the full HDF5 into memory.
with h5py.File(H5, 'r') as h5:
    subjects = sorted(h5['subjects'].keys())
    stored_trials = sum(len(h5['subjects'][sid]['trials']) for sid in subjects)

    for sid in subjects:
        for tid in sorted(h5['subjects'][sid]['trials'].keys()):
            if not (tid.startswith('T') and tid[1:].isdigit() and 1 <= int(tid[1:]) <= 15):
                continue
            uid = f'{sid}_{tid}'
            trial = h5['subjects'][sid]['trials'][tid]
            before = len(issues)
            n = trial[CHANNELS[0]].shape[0] if CHANNELS[0] in trial and trial[CHANNELS[0]].ndim == 2 else 0
            angles_ok = True
            nonfinite_angles = 0
            for channel in CHANNELS:
                if channel not in trial or trial[channel].shape != (n, 101):
                    flag_problem(uid, 'angle_shape', f'{channel}: expected ({n}, 101)')
                    angles_ok = False
                    continue
                values = np.asarray(trial[channel][()])
                if not real_numeric(values):
                    flag_problem(uid, 'angle_type', f'{channel}: expected real numeric data')
                    angles_ok = False
                    continue
                bad = int(np.count_nonzero(~np.isfinite(values)))
                nonfinite_angles += bad
                if bad:
                    flag_problem(uid, 'nonfinite_angles', f'{channel}: {bad} nonfinite entries')
                    angles_ok = False

            target_ok = False
            if 'metcost_mean' not in trial:
                flag_problem(uid, 'target', 'metcost_mean missing')
            else:
                target = np.asarray(trial['metcost_mean'][()])
                if target.size != 1 or not real_numeric(target):
                    flag_problem(uid, 'target', 'metcost_mean must contain one real numeric value')
                else:
                    value = float(target.reshape(-1)[0])
                    target_ok = bool(np.isfinite(value) and value > 0)
                    if not target_ok:
                        flag_problem(uid, 'target', 'metcost_mean is not finite and positive')

            indices_ok = False
            if 'original_cycle_number' not in trial:
                flag_problem(uid, 'cycle_index', 'original_cycle_number missing')
            else:
                indices = np.asarray(trial['original_cycle_number'][()]).reshape(-1)
                if indices.size == n and real_numeric(indices) and np.all(np.isfinite(indices)):
                    indices = indices.astype(np.float64)
                    indices_ok = bool(np.all(indices == np.floor(indices)) and
                                      np.all(np.diff(indices) > 0))
                if not indices_ok:
                    flag_problem(uid, 'cycle_index', 'indices must be finite increasing integers with N entries')

            for attr in ('valid', 'comparison_eligible', 'unrestricted_comparison_eligible'):
                if scalar_attr(trial, attr) != 1:
                    flag_problem(uid, 'eligibility_flag', f'{attr} must equal 1')
            for attr, expected in (('cleaned_cycle_count', n),
                                   ('number_of_cycles', n),
                                   ('number_of_gait_points', 101)):
                actual = scalar_attr(trial, attr)
                if actual is not None and actual != expected:
                    flag_problem(uid, 'count_attribute', f'{attr}: expected {expected}, found {actual}')
            if 'metcost' not in trial:
                flag_problem(uid, 'metabolic_series', 'metcost dataset missing')
            elif scalar_attr(trial, 'number_of_metabolic_samples') is not None:
                if trial['metcost'].size != scalar_attr(trial, 'number_of_metabolic_samples'):
                    flag_problem(uid, 'metabolic_series', 'metcost sample count disagrees with attribute')

            ledger.append({
                'subject_id': sid, 'trial_id': tid, 'trial_uid': uid,
                'outer_fold': FOLD.get(sid, ''), 'retained_cycles': n,
                'population': 'primary_284' if n >= 100 else 'sensitivity_only',
                'angle_values_ok': int(angles_ok), 'nonfinite_angle_values': nonfinite_angles,
                'target_finite_positive': int(target_ok),
                'cycle_index_ok': int(indices_ok),
                'all_checks_ok': int(len(issues) == before),
            })

# Verify age/height on the separate biological source; no biological fields become gait inputs.
with BIO.open('r', newline='', encoding='utf-8-sig') as file:
    bio = list(csv.DictReader(file))
bio_ids = [row.get('subject_id', '') for row in bio]
if len(bio) != 21 or len(set(bio_ids)) != 21 or sorted(bio_ids) != subjects:
    flag_problem('BIO_CSV', 'biological_join', 'expected 21 unique subject IDs matching HDF5')
for row in bio:
    sid = row.get('subject_id', 'UNKNOWN')
    for name in ('age_years', 'height_m'):
        try:
            value = float(row[name])
            if not np.isfinite(value) or value <= 0:
                raise ValueError('nonpositive/nonfinite')
        except (ValueError, KeyError, TypeError):
            flag_problem(sid, 'biological_value', f'{name} must be finite and positive')

primary = [row for row in ledger if row['population'] == 'primary_284']
fold_counts = [sum(row['outer_fold'] == i for row in primary) for i in range(1, 6)]
expected_subjects = [f'S{i:02d}' for i in range(1, 22)]
expected_sensitivity = [row for row in ledger if row['population'] == 'sensitivity_only']
checks = {
    'corrected_hdf5_sha256': source_sha == EXPECTED_SHA,
    '21_expected_subjects': subjects == expected_subjects,
    '559_stored_trials': stored_trials == 559,
    '285_unrestricted_trials': len(ledger) == 285,
    '284_primary_trials': len(primary) == 284,
    '60757_primary_cycles': sum(row['retained_cycles'] for row in primary) == 60757,
    'five_outer_fold_trial_counts': fold_counts == [55, 53, 56, 65, 55],
    'S06_T11_sensitivity_only_85_cycles': len(expected_sensitivity) == 1 and
        expected_sensitivity[0]['trial_uid'] == 'S06_T11' and
        expected_sensitivity[0]['retained_cycles'] == 85,
    'all_unrestricted_trial_numeric_checks': all(row['all_checks_ok'] for row in ledger),
    'biological_data_integrity': not any(row['check'].startswith('biological') for row in issues),
}
for key, passed in checks.items():
    if not passed:
        flag_problem('GLOBAL', key, 'expected source invariant did not pass')

summary = {
    'status': 'PASS' if all(checks.values()) else 'REVIEW_REQUIRED',
    'checks': checks,
    'observed': {
        'subjects': len(subjects), 'stored_trials': stored_trials,
        'unrestricted_trials': len(ledger), 'primary_trials': len(primary),
        'primary_cycles': sum(row['retained_cycles'] for row in primary),
        'primary_outer_fold_trial_counts': fold_counts,
        'nonfinite_angle_values': sum(row['nonfinite_angle_values'] for row in ledger),
        'issues': len(issues),
    },
    'source_sha256': source_sha,
    'biological_csv_sha256': sha256_file(BIO),
    'environment': {
        'python': platform.python_version(),
        'numpy': np.__version__, 'h5py': h5py.__version__,
    },
    'scope_note': 'Checks all 285 unrestricted trials, including S06_T11. '
                  'No model fitting, trial-mean features, prediction metrics, '
                  'or raw metabolic targets are exported. '
                  'Does not independently validate physiological target derivation.',
}

save_csv(OUT / 'trial_numeric_checks.csv', ledger,
         ['subject_id', 'trial_id', 'trial_uid', 'outer_fold', 'retained_cycles',
          'population', 'angle_values_ok', 'nonfinite_angle_values',
          'target_finite_positive', 'cycle_index_ok', 'all_checks_ok'])
save_csv(OUT / 'issues.csv', issues, ['trial_uid', 'check', 'detail'])
(OUT / 'preflight_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

archive = OUT.with_suffix('.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as file:
    for name in ('preflight_summary.json', 'trial_numeric_checks.csv', 'issues.csv'):
        file.write(OUT / name, arcname=f'{OUT.name}/{name}')

print('Numerical preflight:', summary['status'])
print('Primary trials:', len(primary), '| Primary cycles:', summary['observed']['primary_cycles'])
print('Outer-fold trial counts:', fold_counts, '| Issues:', len(issues))
if issues:
    print('First issue:', issues[0])
print('Results folder:', OUT)
print('Results ZIP:', archive)
if summary['status'] != 'PASS':
    print('STOP: review the issues before implementing or training any models.')
